# 06 — Model Validation

Three statistical tests validate the PSI model:

1. **Classification**: Can PSI distinguish successful from blocked sites?
2. **Social resistance test**: Mann-Whitney U — do blocked sites have higher resistance scores?
3. **ROC curve + AUC**: Overall discriminative power
4. **Sensitivity analysis**: Kendall's τ rank stability across weight scenarios

In [ ]:
import sys
sys.path.insert(0, '../..')

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
# Load PSI scores
scores = pd.read_parquet('../../data/processed/psi_scores.parquet')
print(f'Scored locations: {len(scores)}')
print(f'PSI range: {scores["psi"].min():.1f} — {scores["psi"].max():.1f}')
scores.head()

In [ ]:
# Run full validation suite
from analytics.validation.validate import validate_model, get_validation_dataset

validation_set = get_validation_dataset()
results = validate_model(scores, validation_set)
print(f'\nValidation results: {results}')

In [ ]:
# Display ROC curve
roc_path = Path('../../outputs/roc_curve.png')
if roc_path.exists():
    from IPython.display import Image, display
    display(Image(filename=str(roc_path)))
else:
    print('ROC curve not yet generated')

In [ ]:
# Sensitivity analysis
import json
from analytics.validation.validate import sensitivity_analysis
from analytics.scoring.psi import DIMENSION_COLS

features = pd.read_parquet('../../data/processed/features.parquet')

weights_path = Path('../../outputs/derived_weights.json')
if weights_path.exists():
    with open(weights_path) as f:
        derived = json.load(f)['weights']
else:
    from analytics.scoring.weights import FALLBACK_WEIGHTS
    derived = FALLBACK_WEIGHTS

sens = sensitivity_analysis(features, DIMENSION_COLS, derived)

In [ ]:
# PSI distribution
fig, ax = plt.subplots(figsize=(10, 4))
scores['psi'].hist(bins=20, ax=ax, color='#22c55e', alpha=0.7, edgecolor='#1a1b23')
ax.axvline(60, color='#eab308', linestyle='--', label='Viability threshold (60)')
ax.axvline(scores['psi'].mean(), color='white', linestyle=':', alpha=0.5, label=f'Mean ({scores["psi"].mean():.1f})')
ax.set_title('PSI Score Distribution')
ax.set_xlabel('Predictive Suitability Index (0–100)')
ax.legend()
plt.tight_layout()
plt.show()

print(f'\nLocations above 60 (viable): {(scores["psi"] >= 60).sum()}')
print(f'Locations below 60 (risky): {(scores["psi"] < 60).sum()}')